# Gross approximations of protein structures using contact maps
## Model xy-VAE : input sequence of amino-acids [0,1], output contact maps [0,1]

In [1]:
date = '20250428_30000'

import pickle as pkl
import math
from scipy.spatial.distance import cdist
import time
import numpy as np
import matplotlib.pyplot as plt
import importlib, sys, os
from scipy.signal import find_peaks

import torch
import torch.nn.functional as F

import random
random.seed(42)

In [2]:
def get_device():
    """
    Returns the best available device for PyTorch operations.
    Priority: CUDA GPU > MPS (Apple Silicon) > CPU
    """
    if torch.cuda.is_available():
        return torch.device("cuda")
    elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
        return torch.device("mps")
    else:
        return torch.device("cpu")

device = get_device()
print(f"Using device: {device}")

Using device: mps


# Import Data

In [3]:
parent_folder = "nanos_networkx_small"
folders = [name for name in os.listdir(parent_folder) 
           if os.path.isdir(os.path.join(parent_folder, name))]

print(len(folders))

3015


In [4]:
cwd = os.getcwd()
print(cwd)

/Users/alexchilton/DataspellProjects/CAS_AML_Final_Project/gradient_optimization


# Protein class definition

In [5]:
if 'classes.proteins' in sys.modules: importlib.reload(sys.modules['classes.proteins'])
from classes.proteins import *

# Database generation

In [8]:
random.seed(42)

PROTEINS = Database()
SUBPROTEINS = Database()

chunk_length = 50

for folder in folders:
    filename = f"{cwd}/{parent_folder}/{folder}/{folder}{folder[-2:]}_atoms.pkl"
    try:
        with open(filename, 'rb') as f:
            try:
                data = pkl.load(f)
                protein = Protein(data)
                PROTEINS.append(protein)
            except Exception as e:
                print(f"{folder}: error loading data - {e}")
    except FileNotFoundError:
        print(f"{folder}: file not found - {filename}")
    except Exception as e:
        print(f"{folder}: unexpected error - {e}")

PROTEINS.randomize

for protein in PROTEINS:
    aas, cas = protein.explode(chunk_length)
    for i in range(len(aas)):
        subprotein = Protein(aas=aas[i], cas=cas[i])
        SUBPROTEINS.append(subprotein)

SUBPROTEINS.randomize

print (f"{SUBPROTEINS.len} items in full")
SUBPROTEINS = SUBPROTEINS[:30000]
print (f"{SUBPROTEINS.len} items in database")

8SFS_nanobody_C: error loading data - Ran out of input
7F9Z_nanobody_N: error loading data - Ran out of input
6RPJ_nanobody_D: error loading data - Ran out of input
7N0R_nanobody_B: error loading data - Ran out of input
9J3J_nanobody_B: error loading data - Ran out of input
8HBG_nanobody_E: error loading data - Ran out of input
7VND_nanobody_D: error loading data - Ran out of input
8Y9U_nanobody_B: error loading data - Ran out of input
7KC9_nanobody_F: error loading data - Ran out of input
6X07_nanobody_B: error loading data - Ran out of input
8DTN_nanobody_G: error loading data - Ran out of input
1RI8_nanobody_A: file not found - /Users/alexchilton/DataspellProjects/CAS_AML_Final_Project/gradient_optimization/nanos_networkx_small/1RI8_nanobody_A/1RI8_nanobody_A_A_atoms.pkl
7QN5_nanobody_F: error loading data - Ran out of input
7WUJ_nanobody_N: error loading data - Ran out of input
5JDS_nanobody_B: error loading data - Ran out of input
8RW9_nanobody_C: error loading data - Ran out of i

In [14]:
import pickle as pkl
import numpy as np
import os
from datasets import Dataset, DatasetDict
from huggingface_hub import HfApi, login
import pandas as pd
from tqdm import tqdm
import shutil
from datetime import datetime

# Function remains the same
def extract_protein_data(proteins_db):
    """Extract data from your PROTEINS Database object"""
    protein_data = []

    for protein in tqdm(proteins_db.proteins, desc="Extracting protein data"):
        protein_dict = {
            'amino_acid_sequence': protein.aa,
            'length': protein.len,
            'c_alpha_coordinates': protein.ca,
            'distance_matrix': protein.D.tolist(),
        }

        if hasattr(protein, 'contact_maps') and len(protein.contact_maps) > 0:
            protein_dict['contact_maps'] = [cmap.tolist() for cmap in protein.contact_maps]
            protein_dict['contact_map_configs'] = protein.contact_maps_config

        protein_data.append(protein_dict)

    return protein_data

# Updated upload function for personal account with local save
def upload_to_personal_hf(proteins_db, dataset_name="protein-contact-maps",
                          username="alexchilton", save_local=True):
    """Upload your protein dataset to personal Hugging Face account with local backup"""
    # Login to Hugging Face
    login()

    # Extract data from your Database
    print("Extracting protein data...")
    protein_data = extract_protein_data(proteins_db)

    # Convert to pandas DataFrame
    df = pd.DataFrame(protein_data)

    # Create a dataset
    dataset = Dataset.from_pandas(df)

    # Save locally before uploading if requested
    if save_local:
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        local_dir = f"local_protein_dataset_{timestamp}"
        print(f"Saving dataset locally to {local_dir}...")

        # Save using multiple formats for flexibility
        dataset.save_to_disk(local_dir)

        # Also save as pickle for backup
        with open(f"{local_dir}_proteins.pkl", 'wb') as f:
            pkl.dump(proteins_db, f)

        # Save the raw data as parquet
        df.to_parquet(f"{local_dir}_data.parquet")

        print(f"Local save complete. Files saved in {local_dir}")

    # Define a README.md content for the dataset card
    readme_content = f"""---
license: mit
task_categories:
- text-generation
- image-generation
tags:
- protein
- contact-map
- structure
- nanobody
---

# Protein Contact Map Dataset

## Dataset Description

This dataset contains protein structures with contact maps and related information from nanobody sequences.

### Dataset Summary

- **Number of proteins:** {len(protein_data)}
- **Source:** Nanobody protein structures (nanos_networkx_small)
- **Created by:** {username}
- **Date:** {datetime.now().strftime("%Y-%m-%d")}

### Dataset Structure

Each protein entry contains:
- `amino_acid_sequence`: List of amino acid names
- `length`: Number of residues
- `c_alpha_coordinates`: List of [x,y,z] coordinates for C-alpha atoms
- `distance_matrix`: Pairwise distance matrix between C-alpha atoms
- `contact_maps`: List of binary contact maps with different distance thresholds
- `contact_map_configs`: Configuration for each contact map (lower/upper bounds)

### Usage

```python
from datasets import load_dataset
dataset = load_dataset("{username}/{dataset_name}")

# Access a protein
protein = dataset['train'][0]
print(f"Length: {{protein['length']}}")
print(f"First 10 residues: {{protein['amino_acid_sequence'][:10]}}")
```

### Citation

If you use this dataset, please cite:
```
@dataset{{protein_contact_maps,
  title={{Nanobody Protein Contact Map Dataset}},
  author={{Alex Chilton}},
  year={{2025}},
  url={{https://huggingface.co/datasets/{username}/{dataset_name}}}
}}
```
"""

    # Upload to Hugging Face
    print(f"Uploading to {username}/{dataset_name}...")
    dataset.push_to_hub(
        f"{username}/{dataset_name}",
        private=False,  # Set to True if you want a private repository
        commit_message="Initial upload of protein contact map dataset"
    )

    # Create and upload the README.md
    api = HfApi()
    api.upload_file(
        path_or_fileobj=readme_content.encode(),
        path_in_repo="README.md",
        repo_id=f"{username}/{dataset_name}",
        repo_type="dataset",
        commit_message="Add dataset card"
    )

    print(f"Successfully uploaded to https://huggingface.co/datasets/{username}/{dataset_name}")

    return dataset

dataset = upload_to_personal_hf(
    PROTEINS,
    dataset_name="nanobody-contact-maps",
    username="alexchilton",
    save_local=True
)

# To load the dataset later
from datasets import load_dataset
downloaded_dataset = load_dataset("alexchilton/nanobody-contact-maps")
print(f"Downloaded dataset has {len(downloaded_dataset['train'])} proteins")

Extracting protein data...


Extracting protein data: 100%|██████████| 2992/2992 [00:08<00:00, 340.38it/s] 


Saving dataset locally to local_protein_dataset_20250504_100515...


Saving the dataset (0/1 shards):   0%|          | 0/2992 [00:00<?, ? examples/s]

Local save complete. Files saved in local_protein_dataset_20250504_100515
Uploading to casuallabs/nanobody-contact-maps...


HfHubHTTPError: (Request ID: Root=1-68171fd4-11fb7779478722bc20502dff;314e79ef-19b2-435c-8ff0-7de0951045db)

403 Forbidden: You don't have the rights to create a dataset under the namespace "casuallabs".
Cannot access content at: https://huggingface.co/api/repos/create.
Make sure your token has the correct permissions.

# Contact maps definition (y data) and reconstruction test

In [ ]:
def get_Configs(all_distances=None, n_ranges=4, mode='percentile'):

    if mode == 'percentile':
        # even distribution of contacts within the ranges, i suppose it will make the training better
        limits = np.percentile(all_distances, [100*i/n_ranges for i in range(n_ranges)] + [99.6]).tolist()

    elif mode == 'distance':
        # even distance ranges
        min_d, max_d = all_distances.min(), all_distances.max()
        std, mean = all_distances.std(), all_distances.mean()
        min_std, max_std = mean-2*std, mean+2*std
        limits = [min_std + i*((max_std-min_std)/n_ranges) for i in range(n_ranges)] + [max_d]
        limits[0] = min_d

    else:
        print ("Define config mode, mode in ['percentile', 'distances']")
        return False
        
    print ('Limits', limits)
    print ('All distances', len(all_distances))

    CONFIGS = []
    for i in range(len(limits)-1):
        CONFIGS.append({'lower':limits[i], 'upper':limits[i+1]})
        
    return CONFIGS

all_distances = [protein.D for protein in SUBPROTEINS]
all_distances = np.array(all_distances)
all_distances = all_distances[all_distances != 0.]

CONFIGS = get_Configs(all_distances, n_ranges=6, mode='percentile')

plt.figure(figsize=(10,4))
plt.hist(all_distances, bins=200)
_ = plt.show()

In [ ]:
if 'functions.misc' in sys.modules: importlib.reload(sys.modules['functions.misc'])
from functions.misc import *

protein = PROTEINS[33]

CONFIGS = get_Configs(all_distances, n_ranges=6, mode='percentile')
cmaps = [protein.add_contact_map(config['lower'], config['upper'], False) for config in CONFIGS]
titles = [f"lower {round(config['lower'],2)} | upper {round(config['upper'],2)}" for config in CONFIGS]
compare_images(cmaps, titles, masking=False)

In [ ]:
importlib.reload(sys.modules['functions.misc'])

from functions.misc import *

orig_cmap_recon_coords = reconstruct_coords_local_plural_maps(cmaps, CONFIGS, sharpness=15., window=25, 
                                                              dim=3, lr=1e-2, max_iter=5000, prints=10)

In [ ]:
if 'functions.graphics' in sys.modules: importlib.reload(sys.modules['functions.graphics'])
from functions.graphics import *

plotly_2_graphs([np.array(protein.ca), orig_cmap_recon_coords], 'Reconstruction test from true contact maps')

In [ ]:
#importlib.reload(sys.modules['classes.VAE'])
importlib.reload(sys.modules['functions.misc'])

from functions.misc import *

for protein in PROTEINS:
    for config in CONFIGS:
        protein.add_contact_map(config['lower'], config['upper'], True)
    
for protein in SUBPROTEINS:
    for config in CONFIGS:
        protein.add_contact_map(config['lower'], config['upper'], True)

# Statistical lookup

In [ ]:
# some useful stats

UNIQUE_AA = set()
for protein in PROTEINS:
    UNIQUE_AA.update(protein.aa)
UNIQUE_AA.update(['NUL'])
print(f"Unique amino acids in database : {len(UNIQUE_AA)}")

MAX_LEN = max([protein.len for protein in SUBPROTEINS])
print(f"Max protein length :             {MAX_LEN}")

all_vals = []
[all_vals.extend(val) for val in protein.ca for protein in SUBPROTEINS]
print(f"All single coords :              {len(all_vals)}")

MAX_Coord, MIN_Coord = max(all_vals), min(all_vals)
print(f"Max, min coords :                {MAX_Coord, MIN_Coord}")

# Training data

In [ ]:
AA_TRAINING_DATA = torch.stack([torch.tensor(protein.get_padded_aa(MAX_LEN, list(UNIQUE_AA)), dtype=torch.float32) for protein in SUBPROTEINS])
print ('AA training data           ', AA_TRAINING_DATA.shape)

CONTACT_TRAINING_DATA = torch.stack([torch.tensor(np.stack(protein.contact_maps, axis=-1), dtype=torch.float32) for protein in SUBPROTEINS])
print ('contact maps training data ', CONTACT_TRAINING_DATA.shape)

In [ ]:
from torch.utils.data import DataLoader, Dataset, TensorDataset


class CustomDataset(TensorDataset):
    def __init__(self, data_input, data_output, transform=None):
        self.input = data_input
        self.output = data_output

    def __len__(self):
        return len(self.input)

    def __getitem__(self, idx):
        x = self.input[idx]
        y = self.output[idx]
        return x, y

def split_train_test(contact_map, SUBPROTEINS):
    temp_train =  contact_map[:int(SUBPROTEINS.len*0.9)]
    temp_test =   contact_map[int(SUBPROTEINS.len*0.9):]
    temp_train =  temp_train.view(temp_train.size(0),-1)
    temp_test =   temp_test.view(temp_test.size(0),-1)
    return (temp_train, temp_test)

def get_dataloader(train_x, train_y, batch_size=32):
    temp_dataset = CustomDataset(train_x, train_y)
    temp_dataloader = DataLoader(temp_dataset, batch_size=batch_size, shuffle=False)
    return temp_dataloader

BATCH_SIZE = 32

TRAIN_TEST_in =  split_train_test(AA_TRAINING_DATA, SUBPROTEINS)
TRAIN_TEST_out = split_train_test(CONTACT_TRAINING_DATA, SUBPROTEINS)
DATALOADERS =    [(get_dataloader(TRAIN_TEST_in[0], TRAIN_TEST_out[0], BATCH_SIZE), 
                   get_dataloader(TRAIN_TEST_in[1], TRAIN_TEST_out[1], BATCH_SIZE))]

print (len(DATALOADERS), len(DATALOADERS[0]))

# Model definition and training classes

In [ ]:
if 'classes.testVAE' in sys.modules: importlib.reload(sys.modules['classes.testVAE'])
if 'functions.XY_training' in sys.modules: importlib.reload(sys.modules['functions.XY_training'])

from classes.testVAE import *
from functions.XY_training import *

# Hyperparameters and Training

In [ ]:
code_size =     128
input_size =    TRAIN_TEST_in[0].shape[-1]
hidden_size =   128
output_size =   TRAIN_TEST_out[0].shape[-1]
channels =      len(CONFIGS)
epochs =        2000
lr =            1e-5
kl_sigma =      1e-5

print ('input_size ', input_size)
print ('output_size', output_size)
print ('')

In [ ]:
Training_pool = []

for i in range(len(DATALOADERS)):
    
    model = XYVAE(input_size=input_size, hidden_size=hidden_size, code_size=code_size, output_size=output_size, channels=channels, device=device)
    model = model.to(device)

    optimizer = torch.optim.RMSprop(model.parameters(), lr=lr, alpha=0.99, momentum=0.9)
    model.apply(model.weights_init)
    
    Trainer = Training(name=f'XY-VAE {i}', model=model, optimizer=optimizer, with_kl=True, device=device)
    Training_pool .append(Trainer)

print (f'Training pool : about to train {len(Training_pool )} different models, this may require some time ...')

In [ ]:
for i, dataloaders in enumerate(DATALOADERS):
    
    train_dl, test_dl = dataloaders
    Trainer = Training_pool[i]
    Trainer.train(train_dl, test_dl, kl_sig=kl_sigma, loss_type='MSE', epochs=epochs, prints=100, verbose=False, loss_enlarge=1.)

    with open(f'models/{date}_trainer_{i}.pkl', 'wb') as f:
        pkl.dump(Trainer, f)

# Evaluation

In [ ]:
Training_pool = []
for i in range(len(CONFIGS)):
    try:
        with open(f'models/{date}_trainer_{i}.pkl', 'rb') as f:
            Training_pool.append(pkl.load(f))
    except:
        continue

In [ ]:
if 'functions.misc' in sys.modules: importlib.reload(sys.modules['functions.misc'])
from functions.misc import *

[trainer.model.eval() for trainer in Training_pool]

loss_fn = torch.nn.MSELoss(reduction='sum')

def Recon(Trainer, sample_orig, sample_y):
    # Get the device from the model
    device = next(Trainer.model.parameters()).device

    # Move data to the appropriate device
    x = sample_orig.to(device)
    y = sample_y.to(device)

    mu, logvar = Trainer.model.encode(x)
    z = Trainer.model.reparameterize(mu, logvar)
    out = Trainer.model.decode(z)

    loss = loss_fn(out, y)
    score = loss.item()

    # Move tensors back to CPU for numpy operations
    Dist_m = out.detach().cpu()

    return (Dist_m, score)
    
def get_DB(idx):

    DB_SUB = Database()

    folder = folders[idx]
    filename = f"{cwd}/{parent_folder}/{folder}/{folder}{folder[-2:]}_atoms.pkl"
    with open(filename, 'rb') as f:
        data = pkl.load(f)
        
    protein = Protein(data)
        
    aas, cas = protein.explode(chunk_length)
    for i in range(len(aas)):
        subprotein = Protein(aas=aas[i], cas=cas[i])
        for config in CONFIGS:
            subprotein.add_contact_map(config['lower'], config['upper'], True)
        DB_SUB.append(subprotein)
    
    print (f"Original protein length", protein.len)
    print (f"{DB_SUB.len} subsequences")
    
    return (protein, DB_SUB)

def get_training_sets(data, idx=0):

    AA_train = torch.stack([torch.tensor(protein.get_padded_aa(chunk_length, list(UNIQUE_AA)), dtype=torch.float32) for protein in data])
    CM_train = torch.stack([torch.tensor(np.stack(protein.contact_maps, axis=-1), dtype=torch.float32) for protein in data])
    print ('contact maps training data ', CM_train.shape)

    TRAIN_x =    AA_train
    TRAIN_out =  CM_train
    TRAIN_x =    TRAIN_x.view(TRAIN_x.size(0),-1)
    TRAIN_out =  TRAIN_out.view(TRAIN_out.size(0),-1)
    
    return (TRAIN_x, TRAIN_out)

def get_protein_average_contact(Dist_m, protein):

    protein_DM = np.zeros((protein.len, protein.len))
    divider_DM =  np.zeros((protein.len, protein.len))
    hard_mask = np.zeros((protein.len, protein.len)).astype(bool)
    
    for i in range(chunk_length):
        Dist_matrix = Dist_m[i].view(chunk_length,chunk_length).detach().numpy()
        
        protein_DM[-chunk_length+i:,-chunk_length+i:] += Dist_matrix[:chunk_length-i,:chunk_length-i]
        divider_DM[-chunk_length+i:,-chunk_length+i:] += 1.
        hard_mask[-chunk_length+i:,-chunk_length+i:] = 1
        
        if i > 0:
            protein_DM[:i,:i] += Dist_matrix[-i:,-i:]
            divider_DM[:i,:i] += 1.
            hard_mask[:i,:i] = 1

            protein_DM[-chunk_length+i:,:i] += Dist_matrix[:chunk_length-i,-i:]
            divider_DM[-chunk_length+i:,:i] += 1.
            hard_mask[-chunk_length+i:,:i] = 1

            protein_DM[:i, -chunk_length+i:] += Dist_matrix[-i:,:chunk_length-i]
            divider_DM[:i, -chunk_length+i:] += 1.
            hard_mask[:i, -chunk_length+i:] = 1
        
    for i in range(chunk_length, len(Dist_m)):
        Dist_matrix = Dist_m[i].view(chunk_length,chunk_length).detach().numpy()
        protein_DM[i-chunk_length:i, i-chunk_length:i] += Dist_matrix
        divider_DM[i-chunk_length:i, i-chunk_length:i] += 1.
        hard_mask[i-chunk_length:i, i-chunk_length:i] = 1
    
    divider_DM[divider_DM == 0.] = 1.
    protein_DM /= divider_DM

    return (protein_DM, hard_mask)

In [ ]:
results = []
trainer = Training_pool[0]

for idx in [10]:
    
    protein, DB_SUB = get_DB(idx)
    original_cmaps = [protein.add_contact_map(config['lower'], config['upper'], False) for config in CONFIGS]
    
    TRAIN_x, TRAIN_out = get_training_sets(DB_SUB)
    Contact_map, scores = Recon(trainer, TRAIN_x, TRAIN_out)
    print (Contact_map.shape)
    Contact_map = Contact_map.reshape(Contact_map.shape[0], chunk_length, chunk_length, len(CONFIGS))
    print (Contact_map.shape)
    
    recon_cmaps, hard_masks = [], []
    for i in range(len(CONFIGS)):
        recon_average_contact, hard_mask = get_protein_average_contact(Contact_map[:,:,:,i].squeeze(), protein)
        recon_cmaps.append(recon_average_contact)
        hard_masks.append(hard_mask)

    item = {'protein': protein, 'original_cmaps': original_cmaps, 'recon_cmaps': recon_cmaps, 'hard_masks': hard_masks} 
    results.append(item)

compare_images(results[0]['original_cmaps'][:4], [f'Original {i}' for i in range(4)], masking=False, masks=results[0]['hard_masks'])
compare_images(results[0]['recon_cmaps'][:4], [f'Reconst {i}' for i in range(4)], masking=False, masks=results[0]['hard_masks'])


# Reconstructions

In [ ]:
def sharpen(x, temperature=0.01):
    x = np.array(x)
    e_x = np.exp(x / temperature)
    return e_x / np.sum(e_x, axis=-1, keepdims=True)
    
def symmetrize(contact_map):
    symmetrical = np.maximum(contact_map, contact_map.T)
    return symmetrical

def regularize(contact_maps, hard_masks):
    stacked = np.stack(contact_maps, axis=2)
    stacked_sharp = np.round(sharpen(stacked))
    regularized = [stacked_sharp[:,:,i] for i in range(len(contact_maps))]
    [np.fill_diagonal(arr, 0) for arr in regularized]
    return regularized

def getMasked(cmap, mask):
    temp = cmap.copy()
    temp[~mask] = 0.
    return temp

In [ ]:
importlib.reload(sys.modules['functions.misc'])
from functions.misc import *

known_mask = hard_masks[0].copy()
#np.fill_diagonal(known_mask, False)

original_contact_maps = results[0]['original_cmaps']
orig_masked = [getMasked(cmap, known_mask) for cmap in original_contact_maps]
orig_cmap_recon_coords_masked = reconstruct_coords_local_plural_maps(orig_masked.copy(), 
                                                              CONFIGS, hard_mask=known_mask, sharpness=15.,
                                                              window=25, dim=3, lr=1e-2, max_iter=7000, prints=10)

recon_contact_maps = results[0]['recon_cmaps']
recon_contact_maps = [symmetrize(contact_map) for contact_map in recon_contact_maps]
recon_contact_maps = regularize(recon_contact_maps, known_mask)
#range_matrix = fill_missing(get_range_matrix(recon_contact_maps, known_mask, CONFIGS), known_mask, CONFIGS)
#filled_recon_cmaps = get_cmaps_from_range_matrix(range_matrix, CONFIGS)

recon_cmap_recon_coords = reconstruct_coords_local_plural_maps(recon_contact_maps, 
                                                               CONFIGS, hard_mask=known_mask, sharpness=15.,
                                                               window=25, dim=3, lr=1e-2, max_iter=7000, prints=10)


In [ ]:
with open('test_coords.pkl', 'wb') as f:
    data = {'recon': recon_cmap_recon_coords, 'orig': orig_cmap_recon_coords}
    pkl.dump(data, f)

In [ ]:
importlib.reload(sys.modules['functions.graphics'])

from functions.graphics import *

plotly_2_graphs([np.array(results[0]['protein'].ca), orig_cmap_recon_coords_masked], 'Reconstruction from true contact maps')
plotly_2_graphs([np.array(results[0]['protein'].ca), recon_cmap_recon_coords], 'Reconstruction from reconstructed contact map')